In [1]:
import pandas as pd
import numpy as np
import sqlite3
import torch
import torch.nn as nn
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# ---- Load data ----
df = pd.read_csv('data/processed/clean_data.csv')
feature_cols = [
    'op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3', 'sensor_4',
    'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9',
    'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15',
    'sensor_16', 'sensor_17', 'sensor_20', 'sensor_21',
    'sensor_2_roll_mean_5', 'sensor_2_roll_std_5'
]

# ---- Load XGBoost ----
xgb_model = XGBClassifier()
xgb_model.load_model('models/xgboost_model.json')

# ---- Load PyTorch ----
class RULPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.network(x)

input_dim = len(feature_cols)
pytorch_model = RULPredictor(input_dim)
pytorch_model.load_state_dict(torch.load('models/pytorch_model.pth'))
pytorch_model.eval()

# ---- Refit scaler ----
reg_scaler = StandardScaler()
reg_scaler.fit(df[feature_cols])

# ---- Connect to database ----
conn = sqlite3.connect('data/engine_logbook.db')
cursor = conn.cursor()

# ---- Redefine predict_and_log ----
def predict_and_log(machine_id, cycle_data_row, cursor, conn):
    X_row = pd.DataFrame([cycle_data_row[feature_cols]], columns=feature_cols)
    xgb_pred = xgb_model.predict(X_row)[0]
    xgb_confidence = xgb_model.predict_proba(X_row)[0][1]
    status = "Failing" if xgb_pred == 1 else "Healthy"

    X_scaled = reg_scaler.transform(X_row)
    X_tensor = torch.FloatTensor(X_scaled)
    with torch.no_grad():
        predicted_rul = pytorch_model(X_tensor).item()

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cursor.execute('''
        INSERT INTO predictions (machine_id, cycle, status, confidence, predicted_rul, timestamp)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (int(machine_id), int(cycle_data_row['time_in_cycles']), status,
          float(xgb_confidence), float(predicted_rul), timestamp))
    conn.commit()

    return status, xgb_confidence, predicted_rul

print("All models, data, and database connection loaded successfully.")

All models, data, and database connection loaded successfully.


In [2]:

import sys
!{sys.executable} -m pip install reportlab

In [3]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
import os

os.makedirs('manuals', exist_ok=True)

manuals_content = {
    "manual_hpc_degradation.pdf": """
HPC (High Pressure Compressor) Degradation - Repair Manual

Symptoms: Elevated core temperature readings, increased fuel consumption,
gradual pressure ratio decline across the compressor stages.

Common Causes:
- Blade erosion due to particulate ingestion
- Seal wear leading to internal air leakage
- Fouling of compressor blades reducing aerodynamic efficiency

Recommended Actions:
1. Perform borescope inspection of HPC blades for erosion or damage
2. Check and clean compressor inlet filters
3. Inspect interstage seals for wear beyond tolerance
4. If blade erosion exceeds 5% chord length, schedule blade replacement
5. Water wash compressor section to remove fouling deposits

Priority: HIGH - HPC degradation directly impacts engine efficiency and
can lead to compressor stall if left unaddressed.
""",
    "manual_fan_degradation.pdf": """
Fan Degradation - Repair Manual

Symptoms: Increased vibration readings, reduced fan efficiency,
abnormal noise during operation, elevated fan speed sensor deviation.

Common Causes:
- Foreign object damage (FOD) to fan blades
- Bearing wear in the fan shaft assembly
- Imbalance due to blade tip erosion

Recommended Actions:
1. Visually inspect fan blades for nicks, dents, or FOD damage
2. Check fan bearing temperature and vibration signature
3. Perform blade balance check - rebalance if deviation exceeds spec
4. Replace damaged blades immediately if crack propagation is detected
5. Verify fan speed sensor calibration matches actual RPM

Priority: HIGH - Fan imbalance can propagate to downstream stages and
cause secondary damage if not corrected promptly.
""",
    "manual_bearing_wear.pdf": """
Bearing and Vibration Related Issues - Repair Manual

Symptoms: Rising vibration sensor readings, unusual mechanical noise,
gradual increase in bearing housing temperature over operational cycles.

Common Causes:
- Lubrication breakdown or insufficient oil flow
- Bearing surface fatigue from prolonged operation
- Misalignment of rotating shaft assembly

Recommended Actions:
1. Check lubrication system pressure and oil quality
2. Tighten the left bearing housing mounting bolts to specified torque
3. Inspect bearing races for pitting or spalling damage
4. Replace bearing if vibration exceeds 2x baseline amplitude
5. Verify shaft alignment within tolerance after any bearing replacement

Priority: MEDIUM-HIGH - Bearing wear can cascade into shaft damage if
ignored across multiple operational cycles.
""",
    "manual_sensor_drift.pdf": """
Sensor Drift and Calibration Issues - Repair Manual

Symptoms: Gradual deviation of sensor readings from expected baseline
without corresponding physical performance change; erratic readings.

Common Causes:
- Sensor calibration drift over operational hours
- Wiring degradation or connector corrosion
- Thermal cycling effects on sensor electronics

Recommended Actions:
1. Cross-reference suspect sensor against redundant sensor readings
2. Perform sensor recalibration per manufacturer specification
3. Inspect wiring harness and connectors for corrosion or damage
4. Replace sensor if drift exceeds 10% of calibrated range
5. Log calibration date for future maintenance tracking

Priority: MEDIUM - Sensor issues can mask real degradation if not
caught, leading to delayed detection of actual mechanical problems.
""",
    "manual_general_maintenance.pdf": """
General Preventive Maintenance Guidelines

Symptoms: Gradual performance decline across multiple sensors without
a single dominant failure signature; combined minor degradations.

Common Causes:
- Cumulative wear across multiple engine subsystems
- Extended operation beyond recommended maintenance interval
- Environmental factors (dust, humidity, temperature extremes)

Recommended Actions:
1. Schedule comprehensive multi-system inspection
2. Review full sensor history for trending patterns across last 50 cycles
3. Cross-check operational settings against normal operating envelope
4. Perform standard preventive maintenance checklist per interval schedule
5. Document findings for trend analysis in predictive maintenance system

Priority: MEDIUM - Early comprehensive inspection prevents multiple
minor issues from compounding into major failure.
"""
}

for filename, content in manuals_content.items():
    filepath = os.path.join('manuals', filename)
    c = canvas.Canvas(filepath, pagesize=letter)
    c.setFont("Helvetica", 10)
    y = 750
    for line in content.strip().split('\n'):
        if y < 50:
            c.showPage()
            c.setFont("Helvetica", 10)
            y = 750
        c.drawString(50, y, line)
        y -= 15
    c.save()

print(f"Created {len(manuals_content)} synthetic repair manuals in manuals/")

Created 5 synthetic repair manuals in manuals/


In [4]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu pypdf

In [5]:
from pypdf import PdfReader
import os

def extract_text_from_pdf(filepath):
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping word-based chunks."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Process all manuals
all_chunks = []
chunk_sources = []  # tracks which manual each chunk came from

manuals_dir = 'manuals'
for filename in os.listdir(manuals_dir):
    if filename.endswith('.pdf'):
        filepath = os.path.join(manuals_dir, filename)
        text = extract_text_from_pdf(filepath)
        chunks = chunk_text(text)
        all_chunks.extend(chunks)
        chunk_sources.extend([filename] * len(chunks))

print(f"Total chunks created: {len(all_chunks)}")
print(f"\nSample chunk from {chunk_sources[0]}:")
print(all_chunks[0][:300])

Total chunks created: 5

Sample chunk from manual_bearing_wear.pdf:
Bearing and Vibration Related Issues - Repair Manual Symptoms: Rising vibration sensor readings, unusual mechanical noise, gradual increase in bearing housing temperature over operational cycles. Common Causes: - Lubrication breakdown or insufficient oil flow - Bearing surface fatigue from prolonged


In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a lightweight, free, local embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert all chunks into vector embeddings
chunk_embeddings = embed_model.encode(all_chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype('float32')

# Build the FAISS index for fast similarity search
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print(f"Embedding dimension: {dimension}")
print(f"Total vectors indexed: {index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dimension: 384
Total vectors indexed: 5


In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a lightweight, free, local embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert all chunks into vector embeddings
chunk_embeddings = embed_model.encode(all_chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype('float32')

# Build the FAISS index for fast similarity search
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print(f"Embedding dimension: {dimension}")
print(f"Total vectors indexed: {index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dimension: 384
Total vectors indexed: 5


In [8]:
def retrieve_relevant_manual(query_text, top_k=1):
    """
    Given a description of the failure symptoms, find the most
    relevant repair manual chunk using semantic similarity search.
    """
    query_embedding = embed_model.encode([query_text]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            'source': chunk_sources[idx],
            'text': all_chunks[idx],
            'distance': dist
        })
    return results

# Test it with a realistic query
test_query = "engine vibration is high and bearing temperature is rising"
results = retrieve_relevant_manual(test_query, top_k=1)

print("Query:", test_query)
print("\nRetrieved manual:", results[0]['source'])
print("Relevance distance:", results[0]['distance'])
print("\nContent:")
print(results[0]['text'][:500])

Query: engine vibration is high and bearing temperature is rising

Retrieved manual: manual_bearing_wear.pdf
Relevance distance: 0.86530614

Content:
Bearing and Vibration Related Issues - Repair Manual Symptoms: Rising vibration sensor readings, unusual mechanical noise, gradual increase in bearing housing temperature over operational cycles. Common Causes: - Lubrication breakdown or insufficient oil flow - Bearing surface fatigue from prolonged operation - Misalignment of rotating shaft assembly Recommended Actions: 1. Check lubrication system pressure and oil quality 2. Tighten the left bearing housing mounting bolts to specified torque 3.


In [9]:
def build_dynamic_query(machine_id, cycle_data_row, df, feature_cols):
    engine_data = df[df['unit_number'] == machine_id]
    baseline = engine_data.iloc[:10][feature_cols].mean()
    current = cycle_data_row[feature_cols]
    
    # Calculate which sensors have deviated the most from normal
    deviation = ((current - baseline) / (baseline.abs() + 1e-6)).abs()
    top_sensors = deviation.sort_values(ascending=False).head(3).index.tolist()
    
    # Map raw sensor IDs to the physical keywords inside your synthetic PDFs
    sensor_mapping = {
        'sensor_2': 'fan efficiency and fan speed',
        'sensor_2_roll_mean_5': 'fan efficiency and fan speed',
        'sensor_3': 'High Pressure Compressor (HPC) core temperature',
        'sensor_4': 'High Pressure Compressor (HPC) pressure decline',
        'sensor_7': 'fan bearing vibration',
        'sensor_8': 'bearing wear and lubrication',
        'sensor_9': 'bearing mechanical noise',
        'sensor_11': 'High Pressure Compressor (HPC) static pressure',
        'sensor_12': 'sensor electronics and calibration drift',
        'sensor_14': 'bearing vibration and mechanical noise',
        'sensor_15': 'bearing vibration and mechanical noise',
        'sensor_17': 'High Pressure Compressor (HPC) temperature',
        'sensor_20': 'sensor electronics and calibration drift',
        'sensor_21': 'sensor electronics and calibration drift'
    }
    
    # Translate the top 3 sensors into plain-English symptoms
    physical_symptoms = [sensor_mapping.get(s, "general subsystem wear") for s in top_sensors]
    symptoms_text = ", ".join(physical_symptoms)
    
    # FAISS will now search for "HPC", "Bearing", or "Fan", ensuring diverse outputs
    return f"Symptoms include: {symptoms_text}"

In [10]:
import pandas as pd

df = pd.read_csv('data/processed/clean_data.csv')
print("Data loaded:", df.shape)

def generate_diagnostic_report(machine_id, cycle_data_row, cursor, conn):
    """
    Full end-to-end diagnostic: predict failure, retrieve relevant manual,
    generate a plain-English repair recommendation.
    """
    # Step 1: Get prediction (reusing your Step 4 function)
    status, confidence, predicted_rul = predict_and_log(machine_id, cycle_data_row, cursor, conn)

    if status == "Healthy":
        return {
            'machine_id': machine_id,
            'status': status,
            'message': f"Machine {machine_id} is operating normally. No action needed."
        }

    # Step 2: Build a query describing WHY it's failing, using top SHAP-relevant sensors
    # For simplicity, we describe general symptoms based on key sensors
    query_text = build_dynamic_query(machine_id, cycle_data_row, df, feature_cols)

    # Step 3: Retrieve the most relevant manual
    retrieved = retrieve_relevant_manual(query_text, top_k=1)[0]

    # Step 4: Extract the top recommended action (first numbered action in the text)
    manual_text = retrieved['text']

    # Step 5: Build the final human-readable report
    report = {
        'machine_id': machine_id,
        'status': status,
        'confidence': f"{confidence:.1%}",
        'predicted_rul': f"{predicted_rul:.1f} cycles",
        'source_manual': retrieved['source'],
        'message': (
            f"Machine {machine_id} flagged as {status} "
            f"(confidence: {confidence:.1%}, ~{predicted_rul:.0f} cycles remaining). "
            f"Diagnosis based on: {retrieved['source'].replace('manual_', '').replace('.pdf', '').replace('_', ' ')}. "
            f"Recommended: {manual_text.split('Recommended Actions:')[1].split('1.')[1].split('2.')[0].strip() if 'Recommended Actions:' in manual_text else 'See full manual for details.'}"
        )
    }
    return report

# Test on a real failing engine
sample_row = df[df['unit_number'] == 1].iloc[-1]
report = generate_diagnostic_report(1, sample_row, cursor, conn)

print("=== DIAGNOSTIC REPORT ===")
for key, value in report.items():
    print(f"{key}: {value}")

Data loaded: (20631, 24)
=== DIAGNOSTIC REPORT ===
machine_id: 1
status: Failing
confidence: 100.0%
predicted_rul: 17.8 cycles
source_manual: manual_general_maintenance.pdf
message: Machine 1 flagged as Failing (confidence: 100.0%, ~18 cycles remaining). Diagnosis based on: general maintenance. Recommended: Schedule comprehensive multi-system inspection


In [11]:
all_reports = []
for eng_id in df['unit_number'].unique():
    engine_data = df[df['unit_number'] == eng_id]
    max_idx = len(engine_data) - 1
    idx = int(max_idx * np.random.uniform(0.1, 0.95))
    sample_row = engine_data.iloc[idx]
    report = generate_diagnostic_report(eng_id, sample_row, cursor, conn)
    all_reports.append(report)

import pandas as pd
import numpy as np
import textwrap
# We intentionally omit 'import datetime' here to use Cell 1's global import

# 1. Generate the reports with a random mixture (10% to 95% of engine life)
all_reports = []
for eng_id in df['unit_number'].unique():
    engine_data = df[df['unit_number'] == eng_id]
    max_idx = len(engine_data) - 1
    idx = int(max_idx * np.random.uniform(0.1, 0.95))
    sample_row = engine_data.iloc[idx]
    
    report = generate_diagnostic_report(eng_id, sample_row, cursor, conn)
    all_reports.append(report)

reports_df = pd.DataFrame(all_reports)

# 2. Print the Industrial SCADA Telemetry Dashboard
print("\n" + "="*110)
print(f" C-MAPSS ASSET HEALTH & PROGNOSTICS TERMINAL | SYSTEM TIME: {datetime.now().strftime('%Y-%m-%d %H:%M:%SZ')}")
print("="*110)

# Force a guaranteed mixture for presentation (3 Failing, 4 Nominal)
failing_sample = reports_df[reports_df['status'] == 'Failing'].head(3)
nominal_sample = reports_df[reports_df['status'] == 'Healthy'].head(4)
demo_df = pd.concat([failing_sample, nominal_sample]).sample(frac=1) # Shuffle them randomly

# Loop through the guaranteed mixed sample
for index, row in demo_df.iterrows():
    # Pad the machine ID to look like a serial number (e.g., 0001)
    m_id = str(row['machine_id']).zfill(4)
    status = str(row['status']).upper()
    
    if status == 'FAILING':
        print(f" [!] ASSET: ENG-{m_id} | STATUS: CRITICAL FAULT DETECTED | CLASSIFIER CONFIDENCE: {row['confidence']}")
        print(f"     +------------------------------------------------------------------------------------------------+")
        print(f"     | PROGNOSTIC RUL : {row['predicted_rul']} (Threshold Warning)")
        print(f"     | RETRIEVED DOC  : {row['source_manual']} (FAISS Semantic Match)")
        
        # Wrap the long RAG message cleanly
        wrapped_msg = textwrap.fill(str(row['message']), width=85)
        msg_lines = wrapped_msg.split('\n')
        print(f"     | DIRECTIVE      : {msg_lines[0]}")
        for line in msg_lines[1:]:
            print(f"     |                  {line}")
            
        print(f"     +------------------------------------------------------------------------------------------------+")
    else:
        print(f" [*] ASSET: ENG-{m_id} | STATUS: NOMINAL                 | PROGNOSTIC RUL: Safe Operating Range")
        
    print("-" * 110)

print("\n [SYSTEM] Telemetry pipeline complete. Fleet status logged to engine_logbook.db.")
print("="*110 + "\n")


 C-MAPSS ASSET HEALTH & PROGNOSTICS TERMINAL | SYSTEM TIME: 2026-10-03 17:22:14Z
 [*] ASSET: ENG-0003 | STATUS: NOMINAL                 | PROGNOSTIC RUL: Safe Operating Range
--------------------------------------------------------------------------------------------------------------
 [*] ASSET: ENG-0002 | STATUS: NOMINAL                 | PROGNOSTIC RUL: Safe Operating Range
--------------------------------------------------------------------------------------------------------------
 [*] ASSET: ENG-0001 | STATUS: NOMINAL                 | PROGNOSTIC RUL: Safe Operating Range
--------------------------------------------------------------------------------------------------------------
 [!] ASSET: ENG-0013 | STATUS: CRITICAL FAULT DETECTED | CLASSIFIER CONFIDENCE: 88.3%
     +------------------------------------------------------------------------------------------------+
     | PROGNOSTIC RUL : 22.9 cycles (Threshold Warning)
     | RETRIEVED DOC  : manual_general_maintenance.pdf (F

In [12]:
import sys
!{sys.executable} -m pip install streamlit